# Fine-Tuning from Zero: labs in Colab

This notebook runs every lab in the course on a free Colab GPU.
**Before you start:** *Runtime → Change runtime type → T4 GPU*. The labs also run on CPU, just more slowly.

Read [Part 2 — Methods](https://github.com/YOUR-USERNAME/fine-tuning-from-zero/blob/main/en/02_methods.md) first. All tickets are fictional, and the scores on 15 test tickets are **not** evidence of real-world quality.

In [ ]:
!git clone -q https://github.com/YOUR-USERNAME/fine-tuning-from-zero.git
%cd fine-tuning-from-zero/labs
!pip install -q -r requirements.txt
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## Lab 1: Audit the data and run the keyword baseline (no model)

Read the counts, the confusion matrix and the **three mistakes**. Can you explain why the rule got each one wrong?

In [ ]:
!python audit_and_baseline.py --output runs/baseline

## Lab 2: Fine-tune a small classifier (DistilBERT)

Watch `eval_loss` fall across epochs. The script keeps the checkpoint with the best **validation** loss and evaluates the test set once at the end.

In [ ]:
!python train_classifier.py --output runs/classifier

In [ ]:
!cat runs/classifier/final_test.json | head -20

### Compare on the same test IDs

Accuracy hides *which* tickets changed. Count the FIXED and BROKEN rows, not just the totals.

In [ ]:
!python compare_predictions.py runs/baseline/test_predictions.jsonl runs/classifier/test_predictions.jsonl

## Lab 3: LoRA supervised fine-tuning of a chat model (Qwen2.5-0.5B-Instruct)

**Step 1: Inspect before you train.** Which tokens count in the loss? Only the assistant JSON and its end-of-turn token should say `TRAIN`.

In [ ]:
!python inspect_sft_example.py | tail -45

**Step 2: Evaluate the untouched base model** with the same prompt we will train on. This is the baseline.

In [ ]:
!python evaluate_sft.py --mode base --output runs/base_eval.json

**Step 3: Train the LoRA adapter.** Only a small fraction of the parameters train. Look for the `trainable params` line.

In [ ]:
!python train_sft_lora.py --output runs/sft_adapter

**Step 4: Reload the adapter in a fresh process and evaluate it** on the identical frozen test tickets.

In [ ]:
!python evaluate_sft.py --mode adapter --adapter runs/sft_adapter --output runs/adapter_eval.json

In [ ]:
import json
for name in ("base", "adapter"):
    d = json.load(open(f"runs/{name}_eval.json"))
    print(f"{name:8}", {k: round(v, 3) for k, v in d.items() if k.endswith("_rate") or k == "category_accuracy" if v is not None})
print()
for p in json.load(open("runs/adapter_eval.json"))["predictions"][:5]:
    print(p["label"], "|", p["raw"])

## Your turn: teach-back

1. Which metric changed most between base and adapter? Is that change about **format** or about **understanding**?
2. Open `runs/adapter_eval.json`. Is every `reason` supported by the ticket? (This needs human review; no script can check it.)
3. Fill in the [model card](https://github.com/YOUR-USERNAME/fine-tuning-from-zero/blob/main/templates/MODEL_CARD.md) using only numbers you saw in *this* notebook.